# Step 8: Deploy the Model to an Endpoint

**SageMaker Unified Studio Component**: Inference Endpoints

**What you'll learn**: Deploy your trained model as a real-time inference endpoint

In [ ]:
import sagemaker
import os
from sagemaker.sklearn import SKLearnModel
from dotenv import load_dotenv

load_dotenv()
bucket_name = os.getenv('BUCKET_NAME')

# Use SageMaker's execution role (recommended for Unified Studio)
try:
    role = sagemaker.get_execution_role()
    print(f"Using SageMaker execution role: {role}")
except ValueError:
    # Fallback to .env file role if running locally
    role = os.getenv('EXECUTION_ROLE')
    print(f"Using .env execution role: {role}")

## Understanding the Inference Script

SageMaker requires an **inference script** (`inference.py`) that defines how to:
1. Load the model
2. Parse incoming requests
3. Make predictions
4. Format the response

Our `inference.py` has 4 key functions:

```python
# inference.py
import joblib
import json
import numpy as np

def model_fn(model_dir):
    """Load model from directory
    
    Called once when the endpoint starts.
    SageMaker extracts model.tar.gz and passes the directory path.
    """
    model = joblib.load(f"{model_dir}/model.pkl")
    return model

def input_fn(request_body, content_type):
    """Parse input request
    
    Called for each prediction request.
    Transforms raw JSON into the feature array our model expects.
    
    Input: {"temperature": 85, "room_temp": 25}
    Output: [[85, 60]]  (temperature, temp_diff)
    """
    if content_type == 'application/json':
        data = json.loads(request_body)
        temp = data['temperature']
        room_temp = data['room_temp']
        temp_diff = temp - room_temp  # Calculate feature
        return np.array([[temp, temp_diff]])
    raise ValueError(f"Unsupported content type: {content_type}")

def predict_fn(input_data, model):
    """Make prediction
    
    Calls the model and returns both the class (0/1) 
    and the probability of overheating.
    """
    prediction = model.predict(input_data)[0]
    probability = model.predict_proba(input_data)[0][1]
    return {'prediction': int(prediction), 'probability': float(probability)}

def output_fn(prediction, accept):
    """Format output
    
    Converts the Python dict to JSON for the HTTP response.
    """
    return json.dumps(prediction), accept
```

**Key insight**: The `input_fn` calculates `temp_diff` on-the-fly. This means API users only need to send `temperature` and `room_temp` - they don't need to know about our internal feature engineering!

## Create the Inference Script

The `%%writefile` magic command saves the cell content to a file:

In [ ]:
%%writefile inference.py
import joblib
import json
import numpy as np

def model_fn(model_dir):
    """Load model from directory"""
    model = joblib.load(f"{model_dir}/model.pkl")
    return model

def input_fn(request_body, content_type):
    """Parse input request"""
    if content_type == 'application/json':
        data = json.loads(request_body)
        temp = data['temperature']
        room_temp = data['room_temp']
        temp_diff = temp - room_temp
        return np.array([[temp, temp_diff]])
    raise ValueError(f"Unsupported content type: {content_type}")

def predict_fn(input_data, model):
    """Make prediction"""
    prediction = model.predict(input_data)[0]
    probability = model.predict_proba(input_data)[0][1]
    return {'prediction': int(prediction), 'probability': float(probability)}

def output_fn(prediction, accept):
    """Format output"""
    return json.dumps(prediction), accept

## Deploy the Endpoint

Now we create a SageMaker model and deploy it to a real-time endpoint:

In [ ]:
from sagemaker.predictor import Predictor
from sagemaker.serializers import JSONSerializer
from sagemaker.deserializers import JSONDeserializer
from botocore.exceptions import ClientError

model_data = f's3://{bucket_name}/models/logistic_regression/model.tar.gz'
endpoint_name = 'machine-overheat-endpoint'

try:
    # Create SKLearn model with our inference script
    sklearn_model = SKLearnModel(
        model_data=model_data,           # Path to model.tar.gz in S3
        role=role,                        # IAM role for SageMaker
        entry_point='inference.py',       # Our inference script
        framework_version='1.2-1',        # scikit-learn version
        py_version='py3'
    )

    # Deploy to an endpoint
    predictor = sklearn_model.deploy(
        initial_instance_count=1,         # Number of instances
        instance_type='ml.t2.medium',     # Instance type (cost-effective for demo)
        endpoint_name=endpoint_name
    )
    print(f"✓ Endpoint deployed: {predictor.endpoint_name}")

except ClientError as e:
    if 'already existing' in str(e).lower() or 'Cannot create already existing' in str(e):
        print(f"Endpoint '{endpoint_name}' already exists. Connecting to it...")
        predictor = Predictor(
            endpoint_name=endpoint_name,
            serializer=JSONSerializer(),
            deserializer=JSONDeserializer()
        )
        print(f"✓ Connected to existing endpoint: {endpoint_name}")
    else:
        raise e

## Test the Endpoint

Let's make some predictions! The API expects:
- `temperature`: Current machine temperature (°C)
- `room_temp`: Ambient room temperature (°C)

The response includes:
- `prediction`: 0 (Normal) or 1 (Overheat)
- `probability`: Likelihood of overheating (0.0 to 1.0)

In [ ]:
# Test with a normal temperature (should NOT overheat)
test_input = {'temperature': 72, 'room_temp': 25}
response = predictor.predict(test_input)

print(f"Input: {test_input}")
print(f"Response: {response}")
print(f"→ Temperature {test_input['temperature']}°C: {'⚠️ OVERHEAT!' if response['prediction'] == 1 else '✓ Normal'}")
print(f"→ Probability of overheat: {response['probability']*100:.1f}%")

In [ ]:
# Test with a high temperature (SHOULD overheat)
test_input = {'temperature': 85, 'room_temp': 25}
response = predictor.predict(test_input)

print(f"Input: {test_input}")
print(f"Response: {response}")
print(f"→ Temperature {test_input['temperature']}°C: {'⚠️ OVERHEAT!' if response['prediction'] == 1 else '✓ Normal'}")
print(f"→ Probability of overheat: {response['probability']*100:.1f}%")

In [ ]:
# Test with borderline temperature (near 80°C threshold)
test_input = {'temperature': 79, 'room_temp': 25}
response = predictor.predict(test_input)

print(f"Input: {test_input}")
print(f"Response: {response}")
print(f"→ Temperature {test_input['temperature']}°C: {'⚠️ OVERHEAT!' if response['prediction'] == 1 else '✓ Normal'}")
print(f"→ Probability of overheat: {response['probability']*100:.1f}%")

## Key Concepts

**Endpoint Architecture**:
```
Client Request → API Gateway → SageMaker Endpoint → inference.py → Response
     ↓                              ↓
{temperature, room_temp}    model_fn → input_fn → predict_fn → output_fn
```

**Cost Considerations**:
- `ml.t2.medium`: ~$0.05/hour (good for demos)
- Production: Consider `ml.m5.large` for better performance
- **Remember**: Endpoints incur charges while running!

**Next step**: Test the endpoint with various scenarios (see notebook 09)

## Cleanup (Optional)

**Important**: Delete the endpoint when done to avoid charges!

In [ ]:
# Uncomment to delete the endpoint:
# predictor.delete_endpoint()
# print("✓ Endpoint deleted")